In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import os

In [ ]:
df = pd.read_csv('./data/spam.csv', encoding='latin-1')

df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
df.columns = ['label', 'message']

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

def clean_text(text):
    text = text.lower()
    text = "".join([char for char in text if char not in string.punctuation])
    tokens = nltk.word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['clean_message'] = df['message'].apply(clean_text)

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

df[['message', 'clean_message', 'label', 'label_encoded']].head()

In [ ]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['clean_message'])
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
clf = MultinomialNB(alpha=1.0) 
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Model trained and predictions made.")

In [ ]:
print("Performance Evaluation Metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
results = []
for alpha in alphas:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train, y_train)
    results.append(model.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(alphas, results, marker='o', linestyle='--', color='teal')
plt.xscale('log')
plt.xlabel('Alpha (Smoothing Parameter)')
plt.ylabel('Accuracy')
plt.title('Impact of Laplace Smoothing on Accuracy')
plt.grid(True)
plt.show()

In [ ]:
from collections import Counter

spam_words = " ".join(df[df['label_encoded']==1]['clean_message']).split()
ham_words = " ".join(df[df['label_encoded']==0]['clean_message']).split()

spam_counts = Counter(spam_words).most_common(20)
ham_counts = Counter(ham_words).most_common(20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

sns.barplot(x=[x[1] for x in spam_counts], y=[x[0] for x in spam_counts], ax=ax1, palette='rocket')
ax1.set_title('Top 20 Words in Spam Messages')

sns.barplot(x=[x[1] for x in ham_counts], y=[x[0] for x in ham_counts], ax=ax2, palette='mako')
ax2.set_title('Top 20 Words in Ham Messages')

plt.tight_layout()
plt.show()

In [ ]:
test_indices = y_test.index
misclassified_indices = test_indices[y_test != y_pred]
misclassified_df = df.loc[misclassified_indices].copy()
misclassified_df['predicted'] = le.inverse_transform(y_pred[y_test != y_pred])
misclassified_df['actual'] = le.inverse_transform(y_test[y_test != y_pred])

print(f"Total misclassified: {len(misclassified_df)}")
misclassified_df[['message', 'actual', 'predicted']].head(10)